In [1]:
import pandas as pd
import numpy as np
from datetime import timedelta

# ============================================================
# 1) Helpers
# ============================================================

def parse_tr_date_series(s: pd.Series) -> pd.Series:
  
    dt = pd.to_datetime(s, format="%d.%m.%Y", errors="coerce")
    if dt.isna().any():
        dt2 = pd.to_datetime(s, dayfirst=True, errors="coerce")
        dt = dt.fillna(dt2)
    return dt

def to_float_tr(s: pd.Series) -> pd.Series:
   
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors="coerce")

    ser = s.astype(str).str.strip()
    ser = ser.replace({"": np.nan, "nan": np.nan, "None": np.nan, "NaT": np.nan})

    has_dot = ser.str.contains(r"\.", regex=True, na=False)
    has_comma = ser.str.contains(",", regex=False, na=False)
    mask_thousands = has_dot & has_comma
    ser.loc[mask_thousands] = ser.loc[mask_thousands].str.replace(".", "", regex=False)

    ser = ser.str.replace(",", ".", regex=False)
    return pd.to_numeric(ser, errors="coerce")

def week_monday(ts: pd.Timestamp) -> pd.Timestamp:
    """Returns the Monday of the week for the given date."""
    return ts - pd.to_timedelta(ts.weekday(), unit="D")

def linear_regression_slope(xs, ys) -> float:
    if len(xs) != len(ys) or len(xs) < 2:
        return 0.0
    x_mean = sum(xs) / len(xs)
    y_mean = sum(ys) / len(ys)
    num = sum((x - x_mean) * (y - y_mean) for x, y in zip(xs, ys))
    den = sum((x - x_mean) ** 2 for x in xs)
    return num / den if den != 0 else 0.0

def pstdev(values) -> float:
    arr = np.asarray(values, dtype=float)
    if arr.size == 0:
        return np.nan
    return float(arr.std(ddof=0))

# ============================================================
# 2) Main function
#    - Daily deduplication
#    - Weekly feature generation
#    - Prev week = week_start - 7 days
#    - Drop ROW if cur≠5 or prev≠5
# ============================================================

def build_fcm_inputs_from_excel(
    excel_path: str = "upu.xlsx",
    sheet_name: str = "RawData",
    output_excel_path: str = "FCM_INPUTS.xlsx",
    trend_slope_scale: float = 100.0,
    required_days_in_week: int = 5,
    require_both_weeks_full: bool = True,  
):
    # -----------------------------
    # 1) Read
    # -----------------------------
    df = pd.read_excel(excel_path, sheet_name=sheet_name)

    required_cols = [
        "Date", "Machine_ID",
        "OEE", "Plant_OEE_Daily", "OPE_OEE",
        "Avail", "Perf", "Qual",
        "Util",
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing column(s) in Excel:{missing}")

    df = df.copy()

    # -----------------------------
    # 2) Transformations
    # -----------------------------
   
    df["Date"] = parse_tr_date_series(df["Date"])
    # parse edilemeyen tarihleri at
    df = df.dropna(subset=["Date", "Machine_ID"]).copy()

    for col in ["OEE", "Plant_OEE_Daily", "OPE_OEE", "Avail", "Perf", "Qual", "Util"]:
        df[col] = to_float_tr(df[col])

  
    df = df[df["Date"].dt.weekday <= 4].copy()


    df["date_only"] = df["Date"].dt.normalize()  # aynı gün için tek tip timestamp
    df["week_start"] = df["Date"].apply(week_monday)

    daily_cols = ["OEE", "Plant_OEE_Daily", "OPE_OEE", "Avail", "Perf", "Qual", "Util"]
    df_daily = (
        df.groupby(["Machine_ID", "date_only", "week_start"], as_index=False)[daily_cols]
          .mean()
          .sort_values(["Machine_ID", "date_only"])
    )

 
    xs = [1, 2, 3, 4, 5]

    def compute_group_features(g: pd.DataFrame) -> pd.Series:
        machine_id, week_start = g.name

        g = g.sort_values("date_only")
        oee_days = g["OEE"].tolist()
        n_days = len(oee_days)

        week_avg = float(np.mean(oee_days)) if n_days else np.nan
        plant_week_avg = float(g["Plant_OEE_Daily"].mean()) if n_days else np.nan
        ope_week_avg = float(g["OPE_OEE"].mean()) if n_days else np.nan

        avail_week_avg = float(g["Avail"].mean()) if n_days else np.nan
        perf_week_avg  = float(g["Perf"].mean()) if n_days else np.nan
        qual_week_avg  = float(g["Qual"].mean()) if n_days else np.nan
        util_week_avg  = float(g["Util"].mean()) if n_days else np.nan

        slope = linear_regression_slope(xs[:n_days], oee_days) if n_days >= 2 else 0.0
        trend_z = slope / float(trend_slope_scale) if trend_slope_scale != 0 else 0.0
        var_std = pstdev(oee_days) if n_days else np.nan

        return pd.Series({
            "Machine_ID": machine_id,
            "week_start": week_start,

            "n_days": n_days,
            "week_avg": week_avg,
            "plant_week_avg": plant_week_avg,
            "ope_week_avg": ope_week_avg,
            "delta_plant": week_avg - plant_week_avg,
            "delta_ope": week_avg - ope_week_avg,

            "avail_week_avg": avail_week_avg,
            "perf_week_avg": perf_week_avg,
            "qual_week_avg": qual_week_avg,
            "util_week_avg": util_week_avg,

            "trend_slope": slope,
            "trend_z": trend_z,
            "var_std": var_std,
            "week_min_date": g["date_only"].min(),
            "week_max_date": g["date_only"].max(),
        })

    weekly = (
        df_daily.groupby(["Machine_ID", "week_start"], sort=False)
                .apply(compute_group_features)
                .reset_index(drop=True)
    )

 
    weekly = weekly.sort_values(["Machine_ID", "week_start"]).reset_index(drop=True)
    weekly["prev_week_start"] = weekly["week_start"] - pd.Timedelta(days=7)

  
    prev = weekly[["Machine_ID", "week_start", "n_days", "week_avg"]].copy()
    prev = prev.rename(columns={
        "week_start": "prev_week_start",
        "n_days": "prev_n_days",
        "week_avg": "prev_week_avg"
    })

    weekly = weekly.merge(
        prev,
        on=["Machine_ID", "prev_week_start"],
        how="left"
    )

 
    weekly["change"] = weekly["week_avg"] - weekly["prev_week_avg"]


    weekly["both_full"] = (
        (weekly["n_days"] == required_days_in_week) &
        (weekly["prev_n_days"] == required_days_in_week)
    )

    if require_both_weeks_full:
        weekly = weekly[weekly["both_full"]].copy()

 
    feature_cols = [
        "delta_ope", "delta_plant",
        "trend_slope", "trend_z",
        "var_std", "change"
    ]

    fcm_long = weekly.melt(
        id_vars=["Machine_ID", "week_start"],
        value_vars=feature_cols,
        var_name="feature",
        value_name="value"
    ).sort_values(
        ["feature", "Machine_ID", "week_start"]
    ).reset_index(drop=True)

    # -----------------------------
    # SUMMARY
    # -----------------------------
    def summary_stats(s: pd.Series) -> pd.Series:
        s2 = s.dropna().astype(float)
        if s2.empty:
            return pd.Series({"count": 0})
        return pd.Series({
            "count": int(s2.shape[0]),
            "mean": float(s2.mean()),
            "std": float(s2.std(ddof=0)),
            "min": float(s2.min()),
            "p01": float(s2.quantile(0.01)),
            "p05": float(s2.quantile(0.05)),
            "p50": float(s2.quantile(0.50)),
            "p95": float(s2.quantile(0.95)),
            "p99": float(s2.quantile(0.99)),
            "max": float(s2.max()),
        })

    summary = pd.DataFrame(
        {col: summary_stats(weekly[col]) for col in feature_cols}
    ).T.reset_index().rename(columns={"index": "feature"})

    # -----------------------------
    # Write to Excel
    # -----------------------------
    with pd.ExcelWriter(output_excel_path, engine="openpyxl") as writer:
        weekly.to_excel(writer, sheet_name="WEEKLY_FEATURES", index=False)
        fcm_long.to_excel(writer, sheet_name="FCM_LONG", index=False)
        summary.to_excel(writer, sheet_name="SUMMARY", index=False)

    print(f"FCM girdileri Excel'e yazıldı: {output_excel_path}")
    print("Sheets: WEEKLY_FEATURES, FCM_LONG, SUMMARY")


if __name__ == "__main__":
    build_fcm_inputs_from_excel(
        excel_path="upu.xlsx",
        sheet_name="RawData",
        output_excel_path="FCM_INPUTS.xlsx",
        trend_slope_scale=100.0,
        required_days_in_week=5,
        require_both_weeks_full=True
    )


FCM girdileri Excel'e yazıldı: FCM_INPUTS.xlsx
Sheets: WEEKLY_FEATURES, FCM_LONG, SUMMARY
